# Milestone 2: Data Quality dan Cleaning

Notebook ini melanjutkan profiling CommonLingua pada Milestone 1. Data mentah berupa dua CSV (`train` dan `validation`) dengan kolom teks, kode bahasa, sumber, identifier, dan metadata. File mentah tidak diubah; hasil bersih disimpan terpisah sebagai Parquet di `data/processed/`.

Keputusan cleaning: `text` dan `lang` wajib tersedia; null pada metadata opsional dipertahankan agar tidak mengarang informasi. Panjang teks dianalisis sebagai outlier, tetapi teks tidak dipotong atau dihapus hanya karena panjang. Deduplikasi memakai kombinasi `lang`, `source`, dan `identifier` yang tidak null; baris tanpa identifier dipertahankan.

In [9]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
DATA_FILES = {
    'train': DATA_DIR / 'commonlingua_train.csv',
    'validation': DATA_DIR / 'commonlingua_validation.csv',
}
OUTPUT_FILES = {
    split: OUTPUT_DIR / f'commonlingua_{split}_clean.parquet'
    for split in DATA_FILES
}

missing_files = [path for path in DATA_FILES.values() if not path.exists()]
if missing_files:
    missing_list = '\n'.join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f'File dataset tidak ditemukan:\n{missing_list}\n'
        'Pastikan CSV tersedia di data/raw/.'
    )

for split, path in DATA_FILES.items():
    print(f'{split}: {path.name} ({path.stat().st_size / 1_000_000_000:.2f} GB)')

train: commonlingua_train.csv (2.06 GB)
validation: commonlingua_validation.csv (0.23 GB)


## 1. Baca Data dan Profiling DuckDB

Polars membaca CSV secara lazy supaya file besar tidak langsung dimuat seluruhnya ke memori. `SUMMARIZE` DuckDB dijalankan langsung pada file untuk melihat statistik setiap kolom.

In [10]:
def scan_dataset(path: Path) -> pl.LazyFrame:
    return pl.scan_csv(
        path,
        infer_schema=False,
        null_values=['', 'NULL', 'null'],
        low_memory=True,
    )


lazy_data = {split: scan_dataset(path) for split, path in DATA_FILES.items()}
connection = duckdb.connect()
profil_duckdb = {}

for split, path in DATA_FILES.items():
    relation = connection.sql(
        'SUMMARIZE SELECT * FROM read_csv_auto(?, strict_mode=false, parallel=false)',
        params=[str(path)],
    )
    profil_duckdb[split] = pl.DataFrame(
        relation.fetchall(),
        schema=relation.columns,
        orient='row',
    )
    print(f'Profil DuckDB {split}:')
    display(profil_duckdb[split])

for split, frame in lazy_data.items():
    print(f'Skema Polars {split}: {frame.collect_schema()}')

Profil DuckDB train:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""text""","""VARCHAR""","""…","""𱎼家话，又叫重安江苗语，是汉藏语系苗瑶语族一種闲话。 张济…",2323149,null,null,null,null,null,2482568,0.00
"""lang""","""VARCHAR""","""abk""","""zul""",350,null,null,null,null,null,2482568,0.00
"""source""","""VARCHAR""","""Anno""","""usc""",117,null,null,null,null,null,2482568,0.00
"""identifier""","""VARCHAR""","""-0xVHB9387w_1""","""（××××）××执异字第××号""",1901901,null,null,null,null,null,2482568,0.56
"""title""","""VARCHAR""",""" DARDUMAR VOA: Yadda Matashiya…","""󠀦󠀢󠀤󠀙󠀜󠀚󠀛-󠀦󠀢󠀤󠀙󠀜󠀚󠀛""",1574942,null,null,null,null,null,2482568,1.90
"""collection""","""VARCHAR""","""Achaemenid Royal Inscriptions …","""usc""",246,null,null,null,null,null,2482568,1.17
"""license""","""VARCHAR""","""CC-BY-SA""","""Various open science""",14,null,null,null,null,null,2482568,1.08
"""open_type""","""VARCHAR""","""Open_Culture""","""Web""",2,null,null,null,null,null,2482568,1.08
"""creator""","""VARCHAR""","""Barat et al.""","""Masoretic""",2,null,null,null,null,null,2482568,99.93


Profil DuckDB validation:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""text""","""VARCHAR""","""  Abstract: T…","""𧒽崗站（粵拼：leoi1 gong1 zaam6，英文：Le…",277830,null,null,null,null,null,272875,0.00
"""lang""","""VARCHAR""","""abk""","""zul""",337,null,null,null,null,null,272875,0.00
"""source""","""VARCHAR""","""Anno""","""reg_docs""",89,null,null,null,null,null,272875,0.00
"""identifier""","""VARCHAR""","""0000213163_34""","""（2021）黔0502民初12481号""",254827,null,null,null,null,null,272875,0.57
"""title""","""VARCHAR""",""" Gaza, sigidadenw tina se ka s…","""𧒽崗站 (地鐵)""",234288,null,null,null,null,null,272875,1.91
"""collection""","""VARCHAR""","""Achaemenid Royal Inscriptions …","""riao/The Royal Inscriptions of…",181,null,null,null,null,null,272875,0.10
"""license""","""VARCHAR""","""CC-BY-SA""","""Various open science""",11,null,null,null,null,null,272875,0.00
"""open_type""","""VARCHAR""","""Open_Culture""","""Web""",2,null,null,null,null,null,272875,0.00
"""creator""","""VARCHAR""","""Barat et al.""","""Masoretic""",2,null,null,null,null,null,272875,99.93


Skema Polars train: Schema({'text': String, 'lang': String, 'source': String, 'identifier': String, 'title': String, 'collection': String, 'license': String, 'open_type': String, 'creator': String, 'date': String})
Skema Polars validation: Schema({'text': String, 'lang': String, 'source': String, 'identifier': String, 'title': String, 'collection': String, 'license': String, 'open_type': String, 'creator': String, 'date': String})


## 2. Missing Values dan Validasi Kolom Wajib

Kolom `text` dan `lang` menjadi kolom wajib. Spasi di awal dan akhir `text`, `lang`, `source`, dan `identifier` dirapikan sebelum validasi; spasi di tengah teks tidak diubah. Null atau string kosong pada `text`/`lang` akan dikeluarkan. Nilai null pada metadata lain dipertahankan karena ketiadaan metadata bukan alasan untuk mengisi nilai tebakan.

In [11]:
KOLOM_WAJIB = ['text', 'lang']
missing_columns = {
    split: [column for column in KOLOM_WAJIB if column not in frame.collect_schema().names()]
    for split, frame in lazy_data.items()
}
invalid_schema = {split: columns for split, columns in missing_columns.items() if columns}
if invalid_schema:
    raise ValueError(f'Kolom wajib tidak ditemukan: {invalid_schema}')

profil_null = {}
for split, frame in lazy_data.items():
    columns = frame.collect_schema().names()
    null_counts = frame.select(
        pl.len().alias('jumlah_baris'),
        *[pl.col(column).null_count().alias(f'{column}_null') for column in columns],
    ).collect()
    profil_null[split] = null_counts
    print(f'Jumlah null {split}:')
    display(null_counts)

KOLOM_TRIM = ['text', 'lang', 'source', 'identifier']
data_normalized = {}
for split, frame in lazy_data.items():
    available_trim_columns = [
        column for column in KOLOM_TRIM
        if column in frame.collect_schema().names()
    ]
    data_normalized[split] = frame.with_columns(
        [pl.col(column).str.strip_chars() for column in available_trim_columns]
    )

data_valid = {}
jumlah_sebelum = {}
jumlah_setelah_missing = {}
for split, frame in data_normalized.items():
    jumlah_sebelum[split] = int(frame.select(pl.len()).collect().item())
    wajib_terisi = pl.all_horizontal(
        [
            pl.col(column).is_not_null()
            & (pl.col(column).str.strip_chars() != '')
            for column in KOLOM_WAJIB
        ]
    )
    data_valid[split] = frame.filter(wajib_terisi)
    jumlah_setelah_missing[split] = int(
        data_valid[split].select(pl.len()).collect().item()
    )
    print(
        f'{split}: dikeluarkan karena text/lang kosong = '
        f'{jumlah_sebelum[split] - jumlah_setelah_missing[split]:,}'
    )

Jumlah null train:


jumlah_baris,text_null,lang_null,source_null,identifier_null,title_null,collection_null,license_null,open_type_null,creator_null,date_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2482568,0,0,0,13836,47122,28989,26693,26693,2480877,56236


Jumlah null validation:


jumlah_baris,text_null,lang_null,source_null,identifier_null,title_null,collection_null,license_null,open_type_null,creator_null,date_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
272875,0,0,0,1558,5214,270,0,0,272694,3228


train: dikeluarkan karena text/lang kosong = 0
validation: dikeluarkan karena text/lang kosong = 0


## 3. Deteksi Outlier Panjang Teks

Dataset ini tidak memiliki kolom numerik kontinu yang sesuai untuk clipping IQR. Panjang karakter `text` dipakai sebagai indikator diagnostik saja. Teks ekstrem tetap dipertahankan karena dapat berupa dokumen yang valid; tidak dilakukan winsorization atau penghapusan baris berdasarkan panjang.

In [12]:
profil_outlier = {}
for split, frame in data_valid.items():
    panjang_teks = frame.select(
        pl.col('text').str.len_chars().alias('panjang_teks')
    )
    kuartil = panjang_teks.select(
        pl.col('panjang_teks').quantile(0.25).alias('q1'),
        pl.col('panjang_teks').quantile(0.75).alias('q3'),
    ).collect()
    q1 = kuartil['q1'][0]
    q3 = kuartil['q3'][0]
    iqr = q3 - q1
    batas_bawah = max(0, q1 - 1.5 * iqr)
    batas_atas = q3 + 1.5 * iqr
    jumlah_outlier = int(
        panjang_teks.select(
            (
                (pl.col('panjang_teks') < batas_bawah)
                | (pl.col('panjang_teks') > batas_atas)
            ).sum().alias('jumlah_outlier')
        ).collect().item()
    )
    profil_outlier[split] = {
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'batas_bawah': batas_bawah,
        'batas_atas': batas_atas,
        'jumlah_outlier': jumlah_outlier,
    }
    print(
        f'{split}: Q1={q1}, Q3={q3}, IQR={iqr}, '
        f'batas=[{batas_bawah}, {batas_atas}], '
        f'teks ekstrem={jumlah_outlier:,} (dipertahankan)'
    )

train: Q1=131.0, Q3=600.0, IQR=469.0, batas=[0, 1303.5], teks ekstrem=240,558 (dipertahankan)
validation: Q1=130.0, Q3=600.0, IQR=470.0, batas=[0, 1305.0], teks ekstrem=26,683 (dipertahankan)


## 4. Deduplikasi

Gunakan kombinasi `lang`, `source`, dan `identifier` sebagai kunci. Duplikat hanya dihapus ketika seluruh kolom kunci terisi; baris tanpa kunci lengkap tetap dipertahankan karena tidak dapat dicocokkan dengan aman.

In [13]:
KOLOM_KUNCI = ['lang', 'source', 'identifier']
data_dedup = {}
jumlah_setelah_dedup = {}

for split, frame in data_valid.items():
    kolom_tidak_ada = [
        column for column in KOLOM_KUNCI
        if column not in frame.collect_schema().names()
    ]
    if kolom_tidak_ada:
        raise ValueError(f'Kolom kunci tidak ditemukan pada {split}: {kolom_tidak_ada}')

    kunci_lengkap = pl.all_horizontal(
        [
            pl.col(column).is_not_null()
            & (pl.col(column).str.strip_chars() != '')
            for column in KOLOM_KUNCI
        ]
    )
    dengan_urutan = frame.with_row_index('_urutan_asli')
    baris_berkunci = (
        dengan_urutan
        .filter(kunci_lengkap)
        .unique(subset=KOLOM_KUNCI, keep='first', maintain_order=True)
    )
    baris_tanpa_kunci = dengan_urutan.filter(~kunci_lengkap)
    data_dedup[split] = (
        pl.concat([baris_berkunci, baris_tanpa_kunci], how='vertical')
        .sort('_urutan_asli')
        .drop('_urutan_asli')
    )
    jumlah_setelah_dedup[split] = int(
        data_dedup[split].select(pl.len()).collect().item()
    )
    jumlah_duplikat = jumlah_setelah_missing[split] - jumlah_setelah_dedup[split]
    print(
        f'{split}: baris sebelum deduplikasi={jumlah_setelah_missing[split]:,}, '
        f'duplikat dihapus={jumlah_duplikat:,}, '
        f'baris sesudah={jumlah_setelah_dedup[split]:,}'
    )

train: baris sebelum deduplikasi=2,482,568, duplikat dihapus=618,065, baris sesudah=1,864,503
validation: baris sebelum deduplikasi=272,875, duplikat dihapus=15,674, baris sesudah=257,201


## 5. Validasi Data Bersih

Pastikan kolom wajib tidak kosong dan tidak ada kunci lengkap yang terduplikasi. Kode bahasa yang tidak mengikuti pola tiga huruf kecil dilaporkan sebagai temuan untuk ditinjau, bukan langsung dihapus.

In [14]:
hasil_validasi = {}
for split, frame in data_dedup.items():
    kolom_tidak_ada = [
        column for column in KOLOM_WAJIB + KOLOM_KUNCI
        if column not in frame.collect_schema().names()
    ]
    if kolom_tidak_ada:
        raise ValueError(f'Kolom validasi tidak ditemukan pada {split}: {kolom_tidak_ada}')

    kunci_lengkap = pl.all_horizontal(
        [
            pl.col(column).is_not_null()
            & (pl.col(column).str.strip_chars() != '')
            for column in KOLOM_KUNCI
        ]
    )
    baris_berkunci = frame.filter(kunci_lengkap)
    statistik_kunci = baris_berkunci.select(
        pl.len().alias('baris_dengan_kunci'),
        pl.struct(KOLOM_KUNCI).n_unique().alias('kunci_unik'),
    ).collect()
    statistik = frame.select(
        pl.len().alias('jumlah_baris'),
        *[
            (
                pl.col(column).is_null()
                | (pl.col(column).str.strip_chars() == '')
            ).sum().alias(f'{column}_kosong')
            for column in KOLOM_WAJIB
        ],
        (~pl.col('lang').str.contains(r'^[a-z]{3}$'))
        .sum()
        .alias('kode_bahasa_nonstandar'),
    ).collect()

    jumlah_baris_berkunci = int(statistik_kunci['baris_dengan_kunci'][0])
    jumlah_kunci_unik = int(statistik_kunci['kunci_unik'][0])
    hasil_validasi[split] = {
        'jumlah_baris': int(statistik['jumlah_baris'][0]),
        'text_kosong': int(statistik['text_kosong'][0]),
        'lang_kosong': int(statistik['lang_kosong'][0]),
        'duplikat_kunci_lengkap': jumlah_baris_berkunci - jumlah_kunci_unik,
        'kode_bahasa_nonstandar': int(statistik['kode_bahasa_nonstandar'][0]),
    }
    print(f'Validasi {split}: {hasil_validasi[split]}')

    assert hasil_validasi[split]['text_kosong'] == 0
    assert hasil_validasi[split]['lang_kosong'] == 0
    assert hasil_validasi[split]['duplikat_kunci_lengkap'] == 0

Validasi train: {'jumlah_baris': 1864503, 'text_kosong': 0, 'lang_kosong': 0, 'duplikat_kunci_lengkap': 0, 'kode_bahasa_nonstandar': 485}
Validasi validation: {'jumlah_baris': 257201, 'text_kosong': 0, 'lang_kosong': 0, 'duplikat_kunci_lengkap': 0, 'kode_bahasa_nonstandar': 0}


## 6. Simpan Data Bersih

Simpan train dan validation ke file Parquet terpisah agar pemisahan data tetap terjaga. `sink_parquet` menulis hasil query lazy tanpa mengumpulkan seluruh dataset ke memori sekaligus.

In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split, frame in data_dedup.items():
    output_path = OUTPUT_FILES[split]
    frame.sink_parquet(output_path, compression='zstd')
    print(
        f'{split}: {jumlah_sebelum[split]:,} -> '
        f'{jumlah_setelah_dedup[split]:,} baris; '
        f'disimpan ke {output_path}'
    )

train: 2,482,568 -> 1,864,503 baris; disimpan ke D:\Semester 7\BigData\tugas1-ImamPrayuda\data\processed\commonlingua_train_clean.parquet
validation: 272,875 -> 257,201 baris; disimpan ke D:\Semester 7\BigData\tugas1-ImamPrayuda\data\processed\commonlingua_validation_clean.parquet


## Catatan Milestone 2

- Baris dengan `text` atau `lang` kosong dikeluarkan; missing value pada metadata opsional dipertahankan.
- Outlier berdasarkan panjang teks dihitung dengan IQR dan dipertahankan tanpa clipping atau penghapusan.
- Deduplikasi menggunakan kombinasi `lang`, `source`, dan `identifier`; baris tanpa kunci lengkap dipertahankan.
- Jumlah baris yang dibersihkan dan hasil validasi dicetak per split oleh sel-sel di atas.
- Hasil akhir disimpan di `data/processed/commonlingua_train_clean.parquet` dan `data/processed/commonlingua_validation_clean.parquet`.